# L30 · 神经网络的裁判 — Softmax 与交叉熵

**目标**：把模型输出的分数 `logits` 转成概率，再用交叉熵计算正确类别对应的损失。

**为什么对 Aurora 重要**：Month 2 关键词分类器和 Month 3 ASR 的输出层都调用 `softmax`，训练循环里的 `cross_entropy` 就是指导梯度更新的那把尺子。


## 本课剧情：审问模型的自信

分类器最后一层输出任意实数，softmax 把它们映射成和为 1 的概率分布。交叉熵只盯住正确类别那一格：`loss = -log(probs[target])`，概率越高，损失越小。这节课把这两步写成 numpy 代码，并通过调整 logits 的数值观察概率集中度和 loss 的联动。


## 1. logits：模型先给分数，不直接给概率

分类器最后一层常输出一组任意实数，叫 `logits`。它们可以是负数，也不需要相加等于 1。softmax 的任务是把这组分数变成概率分布。

**为什么 softmax 要先减最大值**：`np.exp(z)` 在 z > 709 时溢出为 `inf`。减去 `max(z)` 后，最大那个指数变成 `exp(0) = 1`，其余更小，不会溢出。这个变换不影响结果——分子分母同除以 `exp(max(z))`，概率不变。

**为什么 cross_entropy 只看真实类别**：训练目标是最大化模型给出正确类别的概率，即最大化 `log P(correct)`，等价于最小化 `-log P(correct)`。其余类别的概率通过 softmax 归一化约束已经被间接压低，不需要单独出现在损失里。

In [ ]:
import numpy as np

logits = np.array([2.0, 1.0, 0.1])
print('logits =', logits)
print('sum(logits) =', logits.sum(), ' 不是概率和')
print('argmax =', int(np.argmax(logits)), ' 分数最高的类别')


## 1. ✏️ 实现 `softmax(z)`（数值稳定版）

`softmax(z)_i = e^{z_i} / Σ e^{z_j}`，结果是和为 1 的概率。

**推理路线**：
1. 第 i 个类的概率 = 该类指数 / 所有类指数之和，直接体现「相对大小」。
2. 直接用 `np.exp(z)` 在 z > 709 时得到 `inf`；改用 `np.exp(z - np.max(z))`，减后最大值为 0，`exp(0) = 1`，其余 ≤ 1，不会溢出。
3. 将偏移后的指数向量除以其总和，得到和恰好为 1 的概率向量。

**参考输入输出**：`z=[2,1,0]` → 减 max 后 `[0,-1,-2]` → `exp=[1.0, 0.368, 0.135]` → 归一化 → `[0.665, 0.245, 0.090]`，三者之和为 1.000。

<details><summary>点击查看参考实现</summary>

```python
e = np.exp(z - np.max(z))
return e / e.sum()
```

</details>

## 实验入口：概率量如何从样本里出现

固定一组 logits，观察 softmax 输出的概率向量如何随 logits 差距的扩大而集中；再固定 target，看 `cross_entropy` 的数值如何随正确类别的概率反向变化。


### 写代码前，先把变量表补完整

写 `softmax` 前明确三件事：
- 输入：`z`，任意实数向量（logits），形状 `(n_classes,)`
- 关键步骤：减去 `max(z)` 防溢出 → 对 `exp(z)` 结果归一化
- 返回：概率向量，各元素 ≥ 0，和恰好为 1


In [ ]:
import numpy as np
def softmax(z):
    z = np.asarray(z, dtype=float)
    # ✏️ TODO: 返回概率分布(和为1)
    return None  # ← 改这里


## 动手观察：分数差距如何改变概率

保持类别顺序不变，只放大 logits 的差距。分数差距越大，softmax 后的概率越集中。


In [ ]:
import numpy as np

base = np.array([2.0, 1.0, 0.1])
for scale in [0.5, 1.0, 2.0, 5.0]:
    z = base * scale
    e = np.exp(z - np.max(z))
    p = e / e.sum()
    print(f'scale={scale:>3}: logits={np.round(z,2)} -> probs={np.round(p,3)}')


## 代码实验：同一个正确类别，不同置信度

交叉熵只读取真实类别那一格的概率。正确类别概率越高，loss 越小。


In [ ]:
# 运行前先完成上面的 softmax(z)
for z in [[2.0, 1.0, 0.1], [5.0, 1.0, 0.1], [1.1, 1.0, 0.9]]:
    p = softmax(z)
    print('logits=', z, 'probs=', np.round(p, 3), 'true-class prob=', round(float(p[0]), 3))


In [ ]:
p = softmax([2.0, 1.0, 0.1])
print('概率:', np.round(p, 3), ' 和 =', round(float(p.sum()), 6))
assert abs(p.sum() - 1.0) < 1e-9, '概率应和为 1'
assert np.argmax(p) == 0, '最大分数应得最大概率'
print('✅ softmax 通过。')

## 2. ✏️ 实现交叉熵 `cross_entropy(probs, true_idx)`

只看真实类别的概率：`loss = −log(probs[true_idx])`。预测越自信且正确，损失越小。

**推理路线**：
1. `probs` 是 softmax 的输出，所有元素 ≥ 0 且和为 1；`true_idx` 是正确类别的下标。
2. 用 `probs[true_idx]` 取出正确类别的概率 p。p 越接近 1 代表模型越自信且正确。
3. 取 `-log(p)`：当 p=1 时损失为 0（完美），p=0.1 时损失约为 2.3，p→0 时损失→∞。一行代码就够。

**参考输入输出**：`probs=[0.665, 0.245, 0.090]`，`true_idx=0` → `loss = -log(0.665) ≈ 0.408`；若 `true_idx=2` → `loss = -log(0.090) ≈ 2.408`。

<details><summary>点击查看参考实现</summary>

```python
return -np.log(probs[true_idx])
```

</details>

In [ ]:
def cross_entropy(probs, true_idx):
    # ✏️ TODO: 返回交叉熵损失
    return None  # ← 改这里


In [ ]:
p = softmax([2.0, 1.0, 0.1])
good = cross_entropy(p, 0)   # 真实类=分数最高的那个 -> 损失小
bad  = cross_entropy(p, 2)   # 真实类=分数最低的那个 -> 损失大
print('预测对时损失:', round(good,3), ' 预测错时损失:', round(bad,3))
assert good < bad
print('✅ 通过：你做出了分类训练的损失函数。')

## 3. Toy classifier：从一行分数到一行 loss

下面把一条样本的三类输出完整跑一遍。真实类别是 `target=0`，所以 loss 只看 softmax 后第 0 类的概率。


In [ ]:
class_names = ['speech', 'music', 'noise']
logits = np.array([3.2, 1.4, -0.5])
target = 0

probs = softmax(logits)
loss = cross_entropy(probs, target)

for name, z, p in zip(class_names, logits, probs):
    print(f'{name:>6}: logit={z:+.2f}  prob={p:.3f}')
print('target =', class_names[target], 'loss =', round(float(loss), 3))


**🔗 Aurora 连接**：Month 2 关键词分类器（`src/aurora/classifier/`）的输出层直接调用 `softmax(z)`，训练循环里的 `cross_entropy(probs, target)` 产出标量损失，梯度从这个标量往回传。Month 3 ASR 在 token 级别逐帧重复相同流程，vocabulary 规模可达数千类，但 `softmax` 和 `cross_entropy` 的逻辑与此处完全相同。

In [ ]:
for scale in [0.5, 1.0, 2.0, 5.0]:
    z = np.array([2.0, 1.0, 0.1]) * scale
    p = softmax(z)
    loss = cross_entropy(p, 0)
    print(f'scale={scale:>3}: p_true={p[0]:.3f}, loss={loss:.3f}')


## 参数实验：只改一个旋钮

把 `true_idx=0` 对应的 logit 从 2 逐步增大到 10，观察 softmax 概率从 0.665 趋近 1.0，cross_entropy 从 0.408 趋近 0——正确类别分数越高，模型越自信，损失越低。

再把所有 logit 同时加 100（`z + 100`），确认 softmax 输出与未加之前完全相同。这正是「减 max」的意义：绝对数值不影响概率分布，只有类别之间的相对差距才决定概率大小。

In [ ]:
for scale in [0.5, 1.0, 2.0, 5.0]:
    z = np.array([2.0, 1.0, 0.1]) * scale
    p = softmax(z)
    loss = cross_entropy(p, 0)
    print(f'scale={scale:>3}: p_true={p[0]:.3f}, loss={loss:.3f}')


## 本课收束

现在能用 `softmax(z)` 把任意 logits 向量转成概率分布，用 `cross_entropy(probs, target)` 得到一个标量损失。两个函数合在一起就是分类器「分数 → 概率 → 损失」的完整流程。Month 2 的关键词分类器会直接复用这套代码，Month 3 的 ASR 在 token 级别按相同方式计算训练损失。下一阶段（梯度下降篇）会用交叉熵对参数的导数指导权重更新，届时这里的 `loss` 标量就是起点。


In [ ]:
# 小检查：同一个实验，样本量越大越稳定
import numpy as np

for n in [30, 300, 3000]:
    rng = np.random.default_rng(42)
    samples = rng.integers(1, 7, size=n)
    estimate = np.mean(samples == 6)
    print(f'n={n:4d} -> P(6)≈{estimate:.3f}')
